# End-to-end DiffCAM with a trained CNN

**Weeks:** 31 · **Core question:** Can we implement DiffCAM using learned image features?

**Prerequisites:** complete preceding notebooks in the curriculum. Each lesson targets 45–90 minutes including exercises.

## Concept and motivation

This notebook connects the learned CNN's feature maps, reference embeddings, regularized direction, and spatial aggregation. It deliberately does not call classifier predictions when calculating DiffCAM weights. Grad-CAM is included as a *different question*: which regions influence the class score?

**Reproduction scope:** core feature-difference algorithm on an offline synthetic dataset, not the published pretrained-network benchmarks.

**Learning goal:** Present a diagram of inputs, embeddings, reference scatter, solved weights, and spatial map.

> All images and patient-like examples in this repository are synthetic teaching artifacts, not clinical data.


## Prepare the environment
Run this cell first. All core examples are offline and reproducible.

In [ ]:
from pathlib import Path
import sys
# Works when launched in the repository, even from a nested notebook directory.
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'diffcam_edu').exists()), None)
if root is None:
    raise RuntimeError('Open Jupyter from the extracted diffcam_curriculum folder, or set working directory to it.')
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


## Guided experiment
Execute the code cells in order. Before running each cell, predict what it will show; afterward, explain what changed and why.

### Experiment 1 of 3

In [ ]:
from scipy.ndimage import zoom
from diffcam_edu import make_cell_images
from diffcam_edu.models import fit_cnn, feature_maps, embeddings_from_maps, gradcam
from diffcam_edu.math import diffcam_weights,diffcam_map,normalize_map
images,y,masks=make_cell_images(300,seed=13)
model,history=fit_cnn(images[:240],y[:240],epochs=8,seed=2)
A=feature_maps(model,images);z=embeddings_from_maps(A)
j=int(np.flatnonzero(y[240:]==1)[0]+240)
ref_ix=np.flatnonzero(y[:240]==0)
w=diffcam_weights(z[j],z[ref_ix],ridge=.1)
diffmap=diffcam_map(A[j],w)
gradmap,gradw=gradcam(model,images[j],1)
print('Target idx:',j,'Number of refs:',len(ref_ix),'Feature size:',len(w))

### Experiment 2 of 3

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(13,3))
for a,img,title in zip(ax,[images[j],masks[j],normalize_map(diffmap),normalize_map(gradmap)],['Input','Simulated region','DiffCAM','Grad-CAM']):
    a.imshow(img,cmap='inferno' if title.endswith('CAM') else 'gray');a.set_title(title);a.axis('off')
plt.show()

### Experiment 3 of 3

In [ ]:
upsampled=zoom(normalize_map(diffmap),np.array(images[j].shape)/np.array(diffmap.shape),order=1)
fig,ax=plt.subplots(figsize=(4,4))
ax.imshow(images[j],cmap='gray');ax.imshow(upsampled,alpha=.45,cmap='inferno');ax.set_title('DiffCAM overlay');ax.axis('off');plt.show()
assert diffmap.shape==A.shape[-2:] and len(w)==A.shape[1]

## Exercises — complete independently
1. Choose a different reference class and observe how the explanation changes despite using the same trained CNN.
2. Calculate the Pearson correlation between flattened DiffCAM and Grad-CAM maps; explain what correlation cannot prove.

## Check your understanding

**Exit ticket:** Present a diagram of inputs, embeddings, reference scatter, solved weights, and spatial map.

Write a short answer and save a screenshot or plot from your experiment.

## Further reading
- [DiffCAM (CVPR 2025)](https://openaccess.thecvf.com/content/CVPR2025/html/Li_DiffCAM_Data-Driven_Saliency_Maps_by_Capturing_Feature_Differences_CVPR_2025_paper.html)
- [PyTorch tutorial hub](https://docs.pytorch.org/tutorials/)
- [Course roadmap](../COURSE_MAP.md)